# 03 — Evaluation Analysis

Visualize and analyse:
- ROUGE + BERTScore results (fine-tuned vs baseline)
- Sample predictions vs ground-truth abstracts
- Live demo: summarize a fresh arXiv paper

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Load evaluation results
with open('../evaluation/results.json') as f:
    results = json.load(f)

with open('../evaluation/comparison.json') as f:
    comparison = json.load(f)

print('Fine-tuned model metrics:')
for k, v in results.items():
    if k not in ('sample_predictions', 'model_dir'):
        print(f'  {k:<25} {v}')

In [ ]:
# Bar chart: Baseline vs Fine-tuned ROUGE
metrics = ['rouge1', 'rouge2', 'rougeL']
baseline_vals = [comparison['baseline'][m] for m in metrics]
finetuned_vals = [comparison['finetuned'][m] for m in metrics]

x = range(len(metrics))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar([i - width/2 for i in x], baseline_vals, width, label='Baseline (flan-t5-base)', color='#7bafd4')
bars2 = ax.bar([i + width/2 for i in x], finetuned_vals, width, label='Fine-tuned (LoRA)', color='#e87040')

ax.set_ylabel('Score')
ax.set_title('ROUGE Scores: Baseline vs Fine-tuned')
ax.set_xticks(list(x))
ax.set_xticklabels([m.upper() for m in metrics])
ax.legend()
ax.bar_label(bars1, fmt='%.1f', padding=3)
ax.bar_label(bars2, fmt='%.1f', padding=3)
ax.set_ylim(0, max(finetuned_vals) * 1.2)
plt.tight_layout()
plt.show()

In [ ]:
# Improvement table
improvement_df = pd.DataFrame({
    'Metric': metrics,
    'Baseline': baseline_vals,
    'Fine-tuned': finetuned_vals,
    'Delta': [comparison['improvement'][m] for m in metrics],
})
improvement_df['Delta'] = improvement_df['Delta'].apply(lambda x: f'+{x:.2f}' if x > 0 else f'{x:.2f}')
print(improvement_df.to_string(index=False))

In [ ]:
# Sample predictions
print('Sample Predictions\n' + '='*80)
for i, sample in enumerate(results.get('sample_predictions', [])[:3], 1):
    print(f'\n[{i}] REFERENCE :\n{sample["reference"]}')
    print(f'\n    PREDICTION:\n{sample["prediction"]}')
    print('-' * 80)

In [ ]:
# Live demo: summarize a fresh arXiv paper
import sys
sys.path.insert(0, '..')
from inference.predict import load_model, summarize

model, tokenizer, device = load_model('../outputs/model')

# A real recent paper abstract (paste any arXiv abstract here)
fresh_paper = """
We introduce Llama 3, a family of large language models optimized for dialogue use cases.
Our models are trained on over 15 trillion tokens of data from publicly available sources.
We demonstrate that Llama 3 models achieve strong performance on standard benchmarks
including MMLU, HumanEval, and GSM8K while being efficient to deploy.
We also describe our approach to safety alignment including supervised fine-tuning,
rejection sampling, proximal policy optimization, and direct preference optimization.
"""

summary = summarize(fresh_paper, model, tokenizer, device)
print(f'Input : {fresh_paper.strip()}')
print(f'Summary: {summary}')